# Train models

In [1]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import numpy as np
import os
import itertools
import subprocess
import time

In [2]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [3]:
def is_job_running(job_name):
    result = subprocess.run(['squeue', '-o', '%.28i %.28j %.28u %R', '-u', 'kemal.inecik'],  capture_output=True, text=True)
    jobs = [[j.strip() for j in i.split()]for i in result.stdout.strip().split('\n')]
    for jobid, jobname, jobuser, jobnode in jobs:
        if job_name == jobname:
            return True
    return False

```
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

```
#SBATCH -J {job_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 1-23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

In [4]:
step_per_epoch = 841922 * 0.25 / 512
epochs = list(itertools.chain(range(16, 52), range(52, 100, 2), range(100, 200, 10), range(200, 400, 20)))
epochs = list(np.arange(0, 16, 0.10)) + epochs
epochs = [round(i, 2) for i in epochs]
steps = [int(i*step_per_epoch) for i in epochs]
np.array(steps), len(steps)

(array([     0,     41,     82,    123,    164,    205,    246,    287,
           328,    369,    411,    452,    493,    534,    575,    616,
           657,    698,    739,    781,    822,    863,    904,    945,
           986,   1027,   1068,   1109,   1151,   1192,   1233,   1274,
          1315,   1356,   1397,   1438,   1479,   1521,   1562,   1603,
          1644,   1685,   1726,   1767,   1808,   1849,   1891,   1932,
          1973,   2014,   2055,   2096,   2137,   2178,   2219,   2261,
          2302,   2343,   2384,   2425,   2466,   2507,   2548,   2589,
          2631,   2672,   2713,   2754,   2795,   2836,   2877,   2918,
          2959,   3000,   3042,   3083,   3124,   3165,   3206,   3247,
          3288,   3329,   3370,   3412,   3453,   3494,   3535,   3576,
          3617,   3658,   3699,   3740,   3782,   3823,   3864,   3905,
          3946,   3987,   4028,   4069,   4110,   4152,   4193,   4234,
          4275,   4316,   4357,   4398,   4439,   4480,   4522, 

In [5]:
job_count = 0
overwrite = False
print(f" - Number of models to be trained: {len(steps)!r}")

cpu_gpu = "gpu"

for model_str in ["scanvi", "scvi"]:
    
    for epoch in steps:
        
        output_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        log_file = os.path.join(logs_directory, f"slurm_out_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.log")
        job_name = f"incr_{cpu_gpu}_{model_str}_{epoch}"
        python_name = f"model_training.py"

        if is_job_running(job_name):
            print(f"Training {job_name!r} on {cpu_gpu!r} keeps going for model {model_str!r} and for epoch {epoch!r}.")
        elif overwrite or not os.path.exists(output_dir_path) or not os.path.isdir(output_dir_path):
            try:
                slurm_script = f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, python_name)} --epoch "{epoch}" --model_str "{model_str}" --overwrite "{overwrite}"
    """
                script_name = os.path.join(logs_directory, f"slurm_job_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.sh")
                with open(script_name, "w") as f:
                    f.write(slurm_script)

                print(f"Submitted job on {cpu_gpu!r} {job_count+1} on {cpu_gpu!r}: {job_name!r} for model {model_str!r} and for epoch {epoch!r}")
                subprocess.run(["sbatch", script_name])
                job_count += 1
            finally:
                # time.sleep(0.05)
                os.remove(script_name)

            assert is_job_running(job_name), f"Training {job_name!r} on {cpu_gpu!r} error: model {model_str!r} and for epoch {epoch!r}."
        else:
            print(f"Model exists for model {model_str!r} and for epoch {epoch!r}")
    # if job_count > 32:
    #     break

print(f" - Number of jobs submitted: {job_count}")

 - Number of models to be trained: 240
Submitted job on 'gpu' 1 on 'gpu': 'incr_gpu_scanvi_0' for model 'scanvi' and for epoch 0
Submitted batch job 33799688
Submitted job on 'gpu' 2 on 'gpu': 'incr_gpu_scanvi_41' for model 'scanvi' and for epoch 41
Submitted batch job 33799689
Submitted job on 'gpu' 3 on 'gpu': 'incr_gpu_scanvi_82' for model 'scanvi' and for epoch 82
Submitted batch job 33799690
Submitted job on 'gpu' 4 on 'gpu': 'incr_gpu_scanvi_123' for model 'scanvi' and for epoch 123
Submitted batch job 33799691
Submitted job on 'gpu' 5 on 'gpu': 'incr_gpu_scanvi_164' for model 'scanvi' and for epoch 164
Submitted batch job 33799692
Submitted job on 'gpu' 6 on 'gpu': 'incr_gpu_scanvi_205' for model 'scanvi' and for epoch 205
Submitted batch job 33799693
Submitted job on 'gpu' 7 on 'gpu': 'incr_gpu_scanvi_246' for model 'scanvi' and for epoch 246
Submitted batch job 33799694
Submitted job on 'gpu' 8 on 'gpu': 'incr_gpu_scanvi_287' for model 'scanvi' and for epoch 287
Submitted batc

In [6]:
1

1